# Overlapping Cluster GCN - Heterogeneous Graph Extension

This notebook runs the heterogeneous graph extension of Overlapping-Cluster-GCN.

**Repository**: https://github.com/mamintoosi-papers-codes/Overlapping-Cluster-GCN-Hetero

### Supported Datasets
- **Homogeneous**: Cora, CiteSeer, PubMed, WikiCS
- **Heterogeneous**: ACM, DBLP, IMDB (via PyG HGBDataset)
- **Synthetic**: SYNTH_HETERO (for quick testing)
- **OGB**: OGB_MAG (large-scale heterogeneous)

## 1. Install Dependencies

In [ ]:
import torch

# Install torch-geometric
try:
    import torch_geometric
    print(f'torch-geometric {torch_geometric.__version__} already installed')
except ModuleNotFoundError:
    TORCH = torch.__version__.split('+')[0]
    CUDA = 'cu' + torch.version.cuda.replace('.', '')
    !pip install torch-scatter -f https://pytorch-geometric.com/whl/torch-{TORCH}+{CUDA}.html
    !pip install torch-sparse -f https://pytorch-geometric.com/whl/torch-{TORCH}+{CUDA}.html
    !pip install torch-geometric
    import torch_geometric

# Install other dependencies
!pip install -q texttable karateclub

import torch_geometric
print(f'PyTorch: {torch.__version__}')
print(f'PyG: {torch_geometric.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

## 2. Clone Repository

In [ ]:
!git clone https://github.com/mamintoosi-papers-codes/Overlapping-Cluster-GCN-Hetero.git
%cd Overlapping-Cluster-GCN-Hetero

## 3. Run Homogeneous Datasets (Cora, CiteSeer, PubMed, WikiCS)

In [ ]:
%%time
import pandas as pd

Report = []
for ds_name in ['CiteSeer', 'Cora', 'PubMed', 'WikiCS']:
    for mc in [x / 10.0 for x in range(1, 11)]:
        !python src/main.py --dataset-name {ds_name} --clustering-overlap True \
            --clustering-method danmf --epochs 10 \
            --membership-closeness {mc} --ds-root ./tmp
        Report.append(ds_report)

df_homo = pd.DataFrame(Report, columns=[
    'Winner Membership Closeness', 'Dataset Name',
    'F-1 Score', 'Overlapped Nodes', 'Run Time'
])
df_homo

## 4. Run Heterogeneous Datasets

In [ ]:
%%time
# Test with synthetic heterogeneous dataset (quick, no download needed)
!python src/main.py --dataset-name SYNTH_HETERO --epochs 5 --cluster-number 3 \
    --clustering-overlap True --ds-root ./tmp

In [ ]:
%%time
# Compare cluster mode vs full_batch mode on SYNTH_HETERO
import time

results = []
for mode in ['cluster', 'full_batch']:
    start = time.time()
    !python src/main.py --dataset-name SYNTH_HETERO --epochs 5 --cluster-number 3 \
        --hetero-training-mode {mode} --ds-root ./tmp
    elapsed = time.time() - start
    results.append({'Mode': mode, 'Time': elapsed})

pd.DataFrame(results)

In [ ]:
%%time
# Run ACM dataset (requires download from Tsinghua Cloud)
# Note: If download fails, use SYNTH_HETERO for testing
try:
    !python src/main.py --dataset-name ACM --epochs 10 --cluster-number 5 \
        --clustering-overlap True --ds-root ./tmp
except Exception as e:
    print(f'ACM download failed: {e}')
    print('Use SYNTH_HETERO or OGB_MAG instead')

In [ ]:
%%time
# Run OGB_MAG dataset (large-scale, may take time to download)
try:
    !python src/main.py --dataset-name OGB_MAG --epochs 5 --cluster-number 10 \
        --ds-root ./tmp
except Exception as e:
    print(f'OGB_MAG failed: {e}')

## 5. Visualization

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

if 'df_homo' in dir() and len(df_homo) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    sns.lineplot(x='Winner Membership Closeness', y='F-1 Score',
                 hue='Dataset Name', data=df_homo, ax=axes[0])
    axes[0].set_title('F-1 Score vs Membership Closeness')

    sns.lineplot(x='Winner Membership Closeness', y='Overlapped Nodes',
                 data=df_homo, ax=axes[1])
    axes[1].set_title('Overlapped Nodes vs Membership Closeness')

    sns.lineplot(x='Winner Membership Closeness', y='Run Time',
                 hue='Dataset Name', data=df_homo, ax=axes[2])
    axes[2].set_title('Run Time vs Membership Closeness')

    plt.tight_layout()
    plt.savefig('results/homo_results.png', dpi=150)
    plt.show()
else:
    print('Run Section 3 first to generate results')

## 6. Download Results

In [ ]:
try:
    from google.colab import files
    if 'df_homo' in dir() and len(df_homo) > 0:
        df_homo.to_excel('results/homo_results.xlsx', index=False)
        files.download('results/homo_results.xlsx')
    print('Results downloaded!')
except ImportError:
    print('Not running in Colab - results saved locally')